In [1]:
# Angka tetap. Setiap undian acak di notebook ini memakai SEED.
SEED = 42


# Modul 14: RAG

Cerita lengkapnya ada di `README.md`. Mode cepat tidak mengunduh model. Teksnya sintetis, di folder `dokumen`. Bukan data asli perusahaan.

Sebelum menjalankan kode: pertanyaan tentang kode rak buku cerita seharusnya mengambil kartu perpustakaan. Pertanyaan tentang harga tiket kereta tidak punya kartu. Jawab dulu, baru jalankan.


In [2]:
# Satu ide: cari folder modul dari root repo.
from pathlib import Path
import importlib.util

di_sini = Path.cwd()
calon = [di_sini, *di_sini.parents]
akar = next(p for p in calon if (p / "phase2-ai" / "modul-14-rag" / "rag.py").is_file())
folder = akar / "phase2-ai" / "modul-14-rag"


In [3]:
# Satu ide: muat fungsi potong tanpa jaringan.
spec = importlib.util.spec_from_file_location("rag14", folder / "rag.py")
rag = importlib.util.module_from_spec(spec)
spec.loader.exec_module(rag)


In [4]:
# Satu ide: baca hanya teks sintetis yang ada di folder dokumen.
dokumen = rag.baca_dokumen(folder / "dokumen")
print("berkas_dokumen", len(dokumen))
assert len(dokumen) == 3


berkas_dokumen 3


In [5]:
# Satu ide: potongan adalah baris isi, tanpa baris catatan.
potongan = rag.buat_potongan(dokumen)
print("jumlah_potongan", len(potongan))
assert len(potongan) == 6


jumlah_potongan 6


In [6]:
# Satu ide: simpan vektor ke folder indeks yang tidak masuk repo.
muatan = rag.bangun_indeks(potongan, SEED, folder / rag.FOLDER_INDEKS)
print("folder_indeks", rag.FOLDER_INDEKS)
print("berkas_indeks", rag.NAMA_BERKAS)
print("vektor_tersimpan", len(muatan["potongan"]))
assert len(muatan["potongan"]) == 6


folder_indeks indeks
berkas_indeks potongan.json
vektor_tersimpan 6


In [7]:
# Satu ide: pengambilan soal rak mengembalikan potongan perpustakaan.
hasil = rag.jawab("kode rak buku cerita", muatan)
print("status_rak", hasil["status"])
print("sumber_rak", hasil["sumber"])
print("skor_rak", f"{hasil['skor']:.6f}")
print("bagi_rak", hasil["bagi"])
print("kutipan_rak", hasil["teks"])
assert hasil["status"] == "bersumber"
assert hasil["sumber"] == "perpustakaan.txt"
assert hasil["skor"] >= rag.AMBANG_SKOR
assert hasil["bagi"] >= rag.AMBANG_BAGI


status_rak bersumber
sumber_rak perpustakaan.txt
skor_rak 0.783885
bagi_rak 4
kutipan_rak Kode rak buku cerita adalah B-12.


In [8]:
# Satu ide: soal tanpa sumber ditandai jawaban mengarang.
jebakan = rag.jawab("harga tiket kereta bandung", muatan)
urut = rag.peringkat("kode rak buku cerita", muatan)
margin = urut[0]["skor"] - urut[1]["skor"]
print("status_jebakan", jebakan["status"])
print("skor_jebakan", f"{jebakan['skor']:.6f}")
print("bagi_jebakan", jebakan["bagi"])
print("selisih_rak", f"{margin:.6f}")
assert jebakan["status"] == "jawaban_mengarang"
assert jebakan["teks"] == ""
assert margin >= 0.15


status_jebakan jawaban_mengarang
skor_jebakan 0.000000
bagi_jebakan 0
selisih_rak 0.195568


In [9]:
# Satu ide: revisi model dipin, dan mode cepat tidak mengunduh.
print("model_embedding", "sentence-transformers/all-MiniLM-L6-v2")
print("kelas_embedding", "AutoModel")
print("revisi_embedding", "1110a243fdf4706b3f48f1d95db1a4f5529b4d41")
print("model_jawaban", "HuggingFaceTB/SmolLM2-135M-Instruct")
print("kelas_jawaban", "AutoModelForCausalLM")
print("revisi_jawaban", "12fd25f77366fa6b3b4b768ec3050bf629380bac")
print("unduh", False)


model_embedding sentence-transformers/all-MiniLM-L6-v2
kelas_embedding AutoModel
revisi_embedding 1110a243fdf4706b3f48f1d95db1a4f5529b4d41
model_jawaban HuggingFaceTB/SmolLM2-135M-Instruct
kelas_jawaban AutoModelForCausalLM
revisi_jawaban 12fd25f77366fa6b3b4b768ec3050bf629380bac
unduh False
